In [ ]:
import os
import sys

sys.path.append(os.getcwd())

In [ ]:
from src.data.build_dataset import build_market_dataset

df = build_market_dataset()
df.head()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df.boxplot(figsize=(10,5))

we can see many assymetric tails and outliers beyond whiskers, which is our motivation to do flexible innovations

# Base VAR Test

In [ ]:
from src.models.base_var import BaseVAR

var = BaseVAR(p=2, intercept=True)
design = var.build_design(df)

print("X shape:", design.X.shape)
print("Y shape:", design.Y.shape)
print("Dates shape:", design.dates.shape)
print("First 10 feature names:", design.feature_names[:10])

# Gaussian VAR w/ OLS

In [3]:
%load_ext autoreload
%autoreload 2

from src.data.build_dataset import build_market_dataset
from src.models.gaussian_var import GaussianVAR
import inspect

print(inspect.getsource(GaussianVAR.fit))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
    def fit(self, data: pd.DataFrame, date_col: str = "date") -> GaussianVARResult:
        design = self.base_var.build_design(data, date_col=date_col)
        X, Y = design.X, design.Y

        if not np.isfinite(X).all():
            raise ValueError("X contains non-finite values.")
        if not np.isfinite(Y).all():
            raise ValueError("Y contains non-finite values.")

        X = np.asarray(X, dtype=np.float64)
        Y = np.asarray(Y, dtype=np.float64)

        # Numerically safer than explicit XtX and XtY
        beta_t, residuals_lstsq, rank, s = np.linalg.lstsq(X, Y, rcond=None)
        beta = beta_t.T   # shape (d, k)

        fitted_values = X @ beta_t
        residuals = Y - fitted_values

        T_eff, d = residuals.shape
        k = X.shape[1]

        sigma = (residuals.T @ residuals) / (T_eff - k)

        self.result_ = GaussianVARResult(
            beta=beta,
        

In [ ]:
%load_ext autoreload
%autoreload 2